# Data pre processing

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
import pandas as pd
import numpy as np
import re

# ============================================================
# 1. LOAD DATA
# ============================================================
# Adjust path and separator as needed
df = pd.read_excel('num_data.xlsx')

print(f"Shape: {df.shape}")
print(f"Columns:\n{df.columns.tolist()}")

# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================
# Strip whitespace, standardise
df.columns = df.columns.str.strip()

# Optional: create shorter, code-friendly column names
# (You can skip this if you prefer the originals)
column_rename = {
    '(1) Tell us about yourself': 'about_yourself',
    '(2)  How long have you been caring? (Years)': 'years_caring',
    '(3)  Do you live alone with the person': 'lives_with_cared_for',
    '(5a)  Does Personal Care': 'does_personal_care',
    '(5b)  Does Domestic Care': 'does_domestic_care',
    '(5c)  Does Physical Care': 'does_physical_care',
    '(5d)  Does Finances': 'does_finances',
    '(5e)  Does Health Care': 'does_health_care',
    '(5f)  Does Emotional Care': 'does_emotional_care',
    '(5g)  Communication': 'does_communication',
    '(6) What essential tasks do you  do?': 'essential_tasks',
    '(7)  Hours a week spent caring': 'hours_per_week',
    '(8)  Important things to your caring role': 'important_things',
    '(9)  What can make your role challenging': 'challenges',
    '(10)  Receiving support with caring role': 'receiving_support',
    '(11)  Your health and well being Score': 'health_wellbeing_score',
    '(12) Your health and well being Reason': 'health_wellbeing_reason',
    '(14)  Health and Wellbeing Actions': 'health_wellbeing_actions',
    '(15)  Please tell us if you are working': 'working_status',
    '(16)  Work Education and Training Score': 'work_education_score',
    '(17) Work Education and Training Reason': 'work_education_reason',
    '(18)  Work and Education Actions': 'work_education_actions',
    '(19)  Your Financial Situation Score': 'finance_score',
    '(20) Your Financial Situation Reason': 'finance_reason',
    '(21)  Welfare benefits received': 'welfare_benefits',
    '(22)  Finance Actions': 'finance_actions',
    '(23)  Time Out Score': 'time_out_score',
    '(24) Time Out Reason': 'time_out_reason',
    '(25)  Time Out Actions': 'time_out_actions',
    '(26)  Other caring and family commitments Score': 'family_commitments_score',
    '(27) Other caring and family commitments Reason': 'family_commitments_reason',
    '(28)  Do you have parental responsibility for any children under 18?': 'has_children_under_18',
    '(29)  Other Caring commitment actions': 'caring_commitment_actions',
    '(30)  Relationships Score': 'relationships_score',
    '(31) Relationships Reason': 'relationships_reason',
    '(32)  Relationships Actions': 'relationships_actions',
    '(33)  What is your housing situation?': 'housing_situation',
    '(34)  Your Home Score': 'home_score',
    '(35) Your Home Reason': 'home_reason',
    '(36)  Home Actions': 'home_actions',
    '(37)  Your Diet Score': 'diet_score',
    '(38) Your Diet Reason': 'diet_reason',
    '(39)  Diet Actions': 'diet_actions',
    '(40)  Your Safety Score': 'safety_score',
    '(41) Your Safety Reason': 'safety_reason',
    '(42)  Safety Actions': 'safety_actions',
    'Summary of agreed actions, Additional Notes and Any signposting onward referrals': 'summary_actions',
    'Payment provided to help with actions?': 'payment_provided',
    'Date of Assessment': 'assessment_date',
    'Location of Carer': 'location',
    'Carer Age': 'carer_age',
    'Gender': 'carer_gender',
    'Carer Disability': 'carer_disability',
    'Ethnicity': 'carer_ethnicity',
    'Marital Status': 'marital_status',
    'Religion': 'religion',
    'Sexual Orientation': 'sexual_orientation',
    'Primary Cared For Age': 'pcf_age',
    'Primary Cared For Relationship': 'pcf_relationship',
    'Primary Cared For Disability': 'pcf_disability',
    'Primary Cared For Gender': 'pcf_gender',
    'Primary Cared For Ethnicity': 'pcf_ethnicity',
    'Carer Age Bin': 'carer_age_bin',
    'Sexual Orientation_binned': 'sexual_orientation_binned',
    'Primary Cared For Age Bin': 'pcf_age_bin',
    'Primary Cared For Relationship_binned': 'pcf_relationship_binned',
    'Primary Cared For Gender_binned': 'pcf_gender_binned',
    'Primary Cared For Ethnicity_binned': 'pcf_ethnicity_binned',
}
df.rename(columns=column_rename, inplace=True)

# ============================================================
# 3. FIX ENCODING ISSUES (Global)
# ============================================================
# The ? replacing ' is pervasive across ALL text columns
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].astype(str).str.replace('?', "'", regex=False)

# ============================================================
# 4. STANDARDISE MISSING VALUES
# ============================================================
# Unify the various representations of "missing"
missing_values = ['nan', 'None', 'none', 'NONE', '', 'Not Provided', 
                  '(blank)', 'blank', 'N/A', 'n/a', 'NA', '#VALUE!']

for col in str_cols:
    df[col] = df[col].replace(missing_values, np.nan)
    df[col] = df[col].str.strip()  # remove leading/trailing whitespace

# ============================================================
# 5. CLEAN THE DISABILITY/CONDITION COLUMNS
# ============================================================
def clean_conditions(series):
    """Clean a multi-select semicolon-delimited condition column."""
    
    def _clean_single(val):
        if pd.isna(val) or val in ['None', 'none', '']:
            return np.nan
        
        # Strip leading/trailing semicolons and whitespace
        val = val.strip('; ')
        
        # Split on semicolons
        items = [x.strip() for x in val.split(';') if x.strip()]
        
        # Standardise each item
        standardised = []
        for item in items:
            # Fix Parkinson's variants
            item = re.sub(r"Parkinson[''`]?s?", "Parkinson's", item, flags=re.IGNORECASE)
            
            # Fix Asperger's variants  
            item = re.sub(
                r"Autism/\s*ADHD/\s*Asperger[''`]?s?",
                "Autism/ADHD/Asperger's",
                item
            )
            
            # Fix Alzheimer's
            item = re.sub(r"Dementia/Alzheimers", "Dementia/Alzheimer's", item)
            
            # Standardise "Hearing impairment/ Deaf"
            item = re.sub(
                r"Hearing impairment/?\s*Deaf",
                "Hearing Impairment/Deaf",
                item
            )
            
            # Standardise "Visual Impairment/ Blindness"
            item = re.sub(
                r"Visual Impairment/?\s*Blindness",
                "Visual Impairment/Blindness",
                item
            )
            
            # Standardise "Learning Disability / Difficulty"
            item = re.sub(
                r"Learning Disability\s*/?\s*Difficulty",
                "Learning Disability/Difficulty",
                item
            )
            
            # Standardise "Long term condition/ Chronic Illness"
            item = re.sub(
                r"Long [Tt]erm [Cc]ondition/?\s*Chronic Illness",
                "Long Term Condition/Chronic Illness",
                item
            )
            
            # Standardise "Sickle Cell and/or Thalassaemia Disorder"
            item = re.sub(
                r"Sickle Cell and/?or Thalassaemia Disorder",
                "Sickle Cell/Thalassaemia",
                item
            )
            
            # Catch free-text anomalies
            item = re.sub(
                r"Other long term condition.*",
                "Other",
                item
            )
            item = re.sub(r"Limited physical ability", "Physical Disability", item)
            
            # Handle "Dementia Parkinson's" (space-separated anomaly)
            if item == "Dementia Parkinson's":
                standardised.extend(["Dementia/Alzheimer's", "Parkinson's"])
                continue
            
            # Handle "Long Term Condition/ Chronic Illness Physical Disability" 
            if "Physical Disability" in item and "Long Term" in item and ';' not in item:
                standardised.extend(["Long Term Condition/Chronic Illness", "Physical Disability"])
                continue
                
            item = item.strip()
            if item and item not in ['None', 'none', '']:
                standardised.append(item)
        
        # Remove duplicates while preserving order
        seen = set()
        unique = []
        for s in standardised:
            if s not in seen:
                seen.add(s)
                unique.append(s)
        
        if not unique:
            return np.nan
        
        return ';'.join(unique)
    
    return series.apply(_clean_single)


# Apply to both condition columns
df['carer_disability'] = clean_conditions(df['carer_disability'])
df['pcf_disability'] = clean_conditions(df['pcf_disability'])

# ============================================================
# 6. PARSE SCORE COLUMNS (extract numeric score)
# ============================================================
score_columns = [
    'health_wellbeing_score', 'work_education_score', 'finance_score',
    'time_out_score', 'family_commitments_score', 'relationships_score',
    'home_score', 'diet_score', 'safety_score'
]

for col in score_columns:
    # Extract the leading number from "3 - Sometimes" → 3
    df[col + '_num'] = pd.to_numeric(
        df[col].astype(str).str.extract(r'(\d+)', expand=False),
        errors='coerce'
    )
    # Extract the label too
    df[col + '_label'] = df[col].astype(str).str.extract(r'\d+\s*-\s*(.*)', expand=False)
    df[col + '_label'] = df[col + '_label'].str.strip()

# ============================================================
# 7. PARSE DATES
# ============================================================
df['assessment_date'] = pd.to_datetime(
    df['assessment_date'], dayfirst=True, errors='coerce'
)

# Extract useful date parts
df['assessment_year'] = df['assessment_date'].dt.year
df['assessment_month'] = df['assessment_date'].dt.month
df['assessment_quarter'] = df['assessment_date'].dt.quarter

# ============================================================
# 8. CLEAN NUMERIC COLUMNS
# ============================================================
df['years_caring'] = pd.to_numeric(df['years_caring'], errors='coerce')
df['hours_per_week'] = pd.to_numeric(df['hours_per_week'], errors='coerce')
df['carer_age'] = pd.to_numeric(df['carer_age'], errors='coerce')
df['pcf_age'] = pd.to_numeric(df['pcf_age'], errors='coerce')

# ============================================================
# 9. CLEAN YES/NO COLUMNS
# ============================================================
yes_no_cols = [
    'lives_with_cared_for', 'does_personal_care', 'does_domestic_care',
    'does_physical_care', 'does_finances', 'does_health_care',
    'does_emotional_care', 'does_communication', 'payment_provided',
    'has_children_under_18'
]

for col in yes_no_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df[col] = df[col].map({'yes': 1, 'no': 0}).astype('Int64')

# ============================================================
# 10. CLEAN CATEGORICAL COLUMNS
# ============================================================
# Gender
df['carer_gender'] = df['carer_gender'].str.strip().str.title()
df['pcf_gender'] = df['pcf_gender'].str.strip().str.title() if 'pcf_gender' in df.columns else None

# Working status
df['working_status'] = df['working_status'].str.strip().str.title()

print("\n✅ Basic cleanup complete.")
print(f"Shape: {df.shape}")
print(f"\nSample carer_disability values (cleaned):")
print(df['carer_disability'].dropna().head(10))
print(f"\nSample pcf_disability values (cleaned):")
print(df['pcf_disability'].dropna().head(10))

Shape: (11769, 69)
Columns:
['ID', '(1) Tell us about yourself', '(2)  How long have you been caring? (Years)', '(3)  Do you live alone with the person', '(5a)  Does Personal Care ', '(5b)  Does Domestic Care', '(5c)  Does Physical Care', '(5d)  Does Finances', '(5e)  Does Health Care', '(5f)  Does Emotional Care', '(5g)  Communication', '(6) What essential tasks do you  do?', '(7)  Hours a week spent caring', '(8)  Important things to your caring role', '(9)  What can make your role challenging', '(10)  Receiving support with caring role ', '(11)  Your health and well being Score', '(12) Your health and well being Reason', '(14)  Health and Wellbeing Actions', '(15)  Please tell us if you are working ', '(16)  Work Education and Training Score', '(17) Work Education and Training Reason', '(18)  Work and Education Actions', '(19)  Your Financial Situation Score', '(20) Your Financial Situation Reason', '(21)  Welfare benefits received', '(22)  Finance Actions', '(23)  Time Out Score', 

/var/folders/82/gh1l5w6j5xsbfpfgn8_05_5c0000gn/T/ipykernel_10661/1701780588.py:98: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include='object').columns



✅ Basic cleanup complete.
Shape: (11769, 90)

Sample carer_disability values (cleaned):
1                                    Arthritis;Diabetes
3                                Autism/ADHD/Asperger's
4                       Arthritis;Mental Health Illness
6                   Long Term Condition/Chronic Illness
7     Long Term Condition/Chronic Illness;Mental Hea...
9                                   Physical Disability
10    Long Term Condition/Chronic Illness;Dementia/A...
13            Mental Health Illness;Physical Disability
18    Long Term Condition/Chronic Illness;Mental Hea...
20                                  Physical Disability
Name: carer_disability, dtype: str

Sample pcf_disability values (cleaned):
2                                               Frailty
5     Long Term Condition/Chronic Illness;Physical D...
6     Long Term Condition/Chronic Illness;Mental Hea...
8         Arthritis;Long Term Condition/Chronic Illness
11                  Long Term Condition/Chronic Ill

In [7]:
# ============================================================
# 11. APPROACH B — CONDITION CATEGORY MAPPING
# ============================================================

# Define the category mapping
CONDITION_CATEGORIES = {
    'Cognitive/Neurological': [
        "Dementia/Alzheimer's", "Acquired Brain Injury", 
        "Parkinson's", "Stroke"
    ],
    'Neurodevelopmental': [
        "Autism/ADHD/Asperger's", "Learning Disability/Difficulty"
    ],
    'Mental Health': [
        "Mental Health Illness", "Alcohol or Drug Dependency"
    ],
    'Musculoskeletal/Mobility': [
        "Arthritis", "Physical Disability", "Frailty"
    ],
    'Metabolic/Endocrine': [
        "Diabetes", "Sickle Cell/Thalassaemia"
    ],
    'Cardiorespiratory': [
        "Heart Condition", "Respiratory Condition", "Long Covid"
    ],
    'Sensory': [
        "Hearing Impairment/Deaf", "Visual Impairment/Blindness",
        "Dual Sensory Impairment"
    ],
    'Cancer': [
        "Cancer"
    ],
    'General Chronic': [
        "Long Term Condition/Chronic Illness"
    ]
}

def assign_categories(condition_string, category_map):
    """
    Given a semicolon-delimited condition string,
    return a semicolon-delimited string of matched categories.
    """
    if pd.isna(condition_string):
        return np.nan
    
    conditions = [c.strip() for c in condition_string.split(';')]
    matched_categories = set()
    
    for category, members in category_map.items():
        if any(cond in members for cond in conditions):
            matched_categories.add(category)
    
    if not matched_categories:
        return 'Other/Unknown'
    
    return ';'.join(sorted(matched_categories))


def make_category_flags(df, condition_col, prefix, category_map):
    """
    Create both:
      - A combined category string column
      - Individual binary flag columns per category
      - A condition count column
      - A category count column
    """
    # Combined category string
    df[f'{prefix}_categories'] = df[condition_col].apply(
        lambda x: assign_categories(x, category_map)
    )
    
    # Binary flag per category
    for category in category_map:
        col_name = f'{prefix}_{category.lower().replace("/", "_").replace(" ", "_")}'
        df[col_name] = df[condition_col].apply(
            lambda x: 1 if pd.notna(x) and any(
                cond in [c.strip() for c in x.split(';')]
                for cond in category_map[category]
            ) else 0
        ).astype('Int64')
    
    # Count of individual conditions
    df[f'{prefix}_condition_count'] = df[condition_col].apply(
        lambda x: len([c for c in x.split(';') if c.strip()]) if pd.notna(x) else 0
    )
    
    # Count of categories
    df[f'{prefix}_category_count'] = df[f'{prefix}_categories'].apply(
        lambda x: len(x.split(';')) if pd.notna(x) else 0
    )
    
    # Multimorbidity flag
    df[f'{prefix}_multimorbid'] = (df[f'{prefix}_condition_count'] >= 2).astype('Int64')
    
    return df


# Apply to CARER disability
df = make_category_flags(df, 'carer_disability', 'carer', CONDITION_CATEGORIES)

# Apply to PRIMARY CARED FOR disability
df = make_category_flags(df, 'pcf_disability', 'pcf', CONDITION_CATEGORIES)

# ============================================================
# 12. VERIFY THE RESULTS
# ============================================================
print("=" * 60)
print("CARER CONDITION CATEGORIES — Sample")
print("=" * 60)
category_cols_carer = [c for c in df.columns if c.startswith('carer_') and 
                       any(cat.lower().replace("/","_").replace(" ","_") in c 
                           for cat in CONDITION_CATEGORIES)]
print(df[['carer_disability', 'carer_categories', 'carer_condition_count', 
           'carer_multimorbid'] + category_cols_carer].head(10))

print("\n" + "=" * 60)
print("PCF CONDITION CATEGORIES — Sample")
print("=" * 60)
category_cols_pcf = [c for c in df.columns if c.startswith('pcf_') and 
                     any(cat.lower().replace("/","_").replace(" ","_") in c 
                         for cat in CONDITION_CATEGORIES)]
print(df[['pcf_disability', 'pcf_categories', 'pcf_condition_count',
           'pcf_multimorbid'] + category_cols_pcf].head(10))

print(f"\n✅ Final shape: {df.shape}")

CARER CONDITION CATEGORIES — Sample
                                    carer_disability  \
0                                                NaN   
1                                 Arthritis;Diabetes   
2                                                NaN   
3                             Autism/ADHD/Asperger's   
4                    Arthritis;Mental Health Illness   
5                                                NaN   
6                Long Term Condition/Chronic Illness   
7  Long Term Condition/Chronic Illness;Mental Hea...   
8                                                NaN   
9                                Physical Disability   

                                  carer_categories  carer_condition_count  \
0                                              NaN                      0   
1     Metabolic/Endocrine;Musculoskeletal/Mobility                      2   
2                                              NaN                      0   
3                               Neurode

In [12]:
df.to_csv("temp.csv")

In [13]:
# ============================================================
# 13. ADDITIONAL DATA INTEGRITY TRANSFORMATIONS
# ============================================================

# ----------------------------------------------------------
# 13a. Nullify underage carers (age < 18)
# ----------------------------------------------------------
underage_count = (df['carer_age'] < 18).sum()
print(f"⚠️  Underage carers found (age < 18): {underage_count}")

df.loc[df['carer_age'] < 18, 'carer_age'] = np.nan

# Also nullify the carer_age_bin for these rows to keep consistency
df.loc[df['carer_age'].isna(), 'carer_age_bin'] = np.nan


# ----------------------------------------------------------
# 13b. Cap 'years_caring' at plausible maximum
# ----------------------------------------------------------
# Logic: years_caring cannot exceed:
#   - carer_age - 18 (must have been an adult when caring started)
#   - pcf_age (can't care for someone longer than they've been alive)
# Take the minimum of all available upper bounds

def cap_years_caring(row):
    yc = row['years_caring']
    
    if pd.isna(yc):
        return np.nan
    
    upper_bounds = []
    
    # Carer must have been 18+ when they started caring
    if pd.notna(row['carer_age']):
        max_from_carer = row['carer_age'] - 18
        if max_from_carer > 0:
            upper_bounds.append(max_from_carer)
        else:
            # Carer is under 18 (already nullified) or exactly 18
            # so max years caring is 0
            upper_bounds.append(0)
    
    # Can't have cared longer than the cared-for person has been alive
    if pd.notna(row['pcf_age']):
        upper_bounds.append(row['pcf_age'])
    
    if upper_bounds:
        cap = min(upper_bounds)
        if yc > cap:
            return cap
        return yc
    
    # No age data to validate against — keep as-is
    return yc

# Track how many get adjusted
original_yc = df['years_caring'].copy()
df['years_caring'] = df.apply(cap_years_caring, axis=1)
adjusted_count = ((original_yc != df['years_caring']) & original_yc.notna()).sum()
print(f"⚠️  years_caring capped for {adjusted_count} rows")


# ============================================================
# 14. FURTHER INTEGRITY CHECKS & ISSUES
# ============================================================

# ----------------------------------------------------------
# 14a. Carer age vs Primary Cared For age — relationship sanity
# ----------------------------------------------------------
# If relationship is "Parent", the cared-for person should 
# generally be OLDER than the carer (you care for your parent)
# If relationship is "Son/Daughter/Child", cared-for should be YOUNGER

def flag_age_relationship_mismatch(row):
    """Flag rows where carer/pcf age doesn't match the stated relationship."""
    rel = row.get('pcf_relationship')
    carer_age = row.get('carer_age')
    pcf_age = row.get('pcf_age')
    
    if pd.isna(rel) or pd.isna(carer_age) or pd.isna(pcf_age):
        return False
    
    rel_lower = str(rel).lower().strip()
    
    # Caring for a parent — parent should generally be older
    if rel_lower in ['parent', 'mother', 'father']:
        if pcf_age < carer_age:
            return True
    
    # Caring for a child — child should generally be younger
    if rel_lower in ['son', 'daughter', 'child', 'son/daughter']:
        if pcf_age > carer_age:
            return True
    
    # Caring for a spouse/partner — extreme age gaps (>40 years) are suspicious
    if rel_lower in ['spouse', 'partner', 'husband', 'wife']:
        if abs(pcf_age - carer_age) > 40:
            return True
    
    return False

df['flag_age_relationship_mismatch'] = df.apply(
    flag_age_relationship_mismatch, axis=1
)
print(f"⚠️  Age-relationship mismatches: {df['flag_age_relationship_mismatch'].sum()}")


# ----------------------------------------------------------
# 14b. Implausible ages
# ----------------------------------------------------------
# Flag extreme ages (e.g., carer aged 5 or 110)
df['flag_carer_age_implausible'] = (
    (df['carer_age'] < 18) | (df['carer_age'] > 105)
)
df['flag_pcf_age_implausible'] = (
    (df['pcf_age'] < 0) | (df['pcf_age'] > 115)
)
print(f"⚠️  Implausible carer ages: {df['flag_carer_age_implausible'].sum()}")
print(f"⚠️  Implausible PCF ages: {df['flag_pcf_age_implausible'].sum()}")


# ----------------------------------------------------------
# 14c. Hours per week > 168 (impossible)
# ----------------------------------------------------------
# There are only 168 hours in a week
df['flag_hours_implausible'] = df['hours_per_week'] > 168
implausible_hours = df['flag_hours_implausible'].sum()
print(f"⚠️  Hours/week > 168: {implausible_hours}")

# Also flag suspiciously round/high values — some surveys cap at 50+
# You may want to investigate values like 100, 150, 168
print(f"   Hours/week distribution:")
print(df['hours_per_week'].describe())


# ----------------------------------------------------------
# 14d. Assessment date outliers
# ----------------------------------------------------------
# Check for dates that seem wrong (far future or far past)
date_min = df['assessment_date'].min()
date_max = df['assessment_date'].max()
print(f"\n📅 Assessment date range: {date_min} to {date_max}")

# Flag dates before a reasonable programme start or after today
reasonable_start = pd.Timestamp('2015-01-01')
today = pd.Timestamp('today')
df['flag_date_outlier'] = (
    (df['assessment_date'] < reasonable_start) | 
    (df['assessment_date'] > today)
)
print(f"⚠️  Date outliers: {df['flag_date_outlier'].sum()}")


# ----------------------------------------------------------
# 14e. Score consistency checks
# ----------------------------------------------------------
# All score columns should be 1-5. Flag anything outside that range.
score_num_cols = [c for c in df.columns if c.endswith('_score_num')]
for col in score_num_cols:
    out_of_range = ((df[col] < 1) | (df[col] > 5)).sum()
    if out_of_range > 0:
        print(f"⚠️  {col}: {out_of_range} values outside 1-5 range")


# ----------------------------------------------------------
# 14f. Carer disability = conditions of the CARER, not the 
#      cared-for person — check for suspicious overlap
# ----------------------------------------------------------
# If carer_disability and pcf_disability are IDENTICAL, it could 
# indicate a copy-paste error during data entry
df['flag_identical_disabilities'] = (
    df['carer_disability'].notna() & 
    df['pcf_disability'].notna() & 
    (df['carer_disability'] == df['pcf_disability'])
)
print(f"⚠️  Identical carer & PCF disabilities: {df['flag_identical_disabilities'].sum()}")


# ----------------------------------------------------------
# 14g. Duplicate records
# ----------------------------------------------------------
# Check for exact duplicates
exact_dupes = df.duplicated().sum()
print(f"⚠️  Exact duplicate rows: {exact_dupes}")

# Check for duplicate IDs (same carer assessed multiple times is valid,
# but true ID duplicates are errors)
id_dupes = df['ID'].duplicated().sum()
print(f"⚠️  Duplicate IDs: {id_dupes}")

# If IDs can legitimately repeat (re-assessments), check for 
# same ID + same date duplicates
if id_dupes > 0:
    id_date_dupes = df.duplicated(subset=['ID', 'assessment_date']).sum()
    print(f"   ↳ Same ID + same date: {id_date_dupes} (likely true duplicates)")


# ----------------------------------------------------------
# 14h. Free-text in structured fields
# ----------------------------------------------------------
# The "Reason" columns sometimes bleed into adjacent columns
# Check if score columns contain unexpected text
for col in score_num_cols:
    null_scores = df[col].isna().sum()
    total = len(df)
    if null_scores / total > 0.3:
        print(f"⚠️  {col}: {null_scores}/{total} nulls — possible parsing issues")


# ----------------------------------------------------------
# 14i. Gender consistency between carer and PCF
# ----------------------------------------------------------
# Not an "error" per se, but check for blanks and unusual values
print(f"\nCarer gender distribution:")
print(df['carer_gender'].value_counts(dropna=False))
print(f"\nPCF gender distribution:")
print(df['pcf_gender'].value_counts(dropna=False) if 'pcf_gender' in df.columns else "Column not found")


# ----------------------------------------------------------
# 14j. Ethnicity consistency
# ----------------------------------------------------------
# Check for rare/inconsistent ethnicity values
print(f"\nCarer ethnicity distribution:")
print(df['carer_ethnicity'].value_counts(dropna=False).head(15))


# ============================================================
# 15. SUMMARY OF ALL FLAGS
# ============================================================
flag_cols = [c for c in df.columns if c.startswith('flag_')]
print("\n" + "=" * 60)
print("DATA QUALITY FLAG SUMMARY")
print("=" * 60)
for col in flag_cols:
    count = df[col].sum()
    pct = count / len(df) * 100
    print(f"  {col}: {count} rows ({pct:.1f}%)")

total_flagged = df[flag_cols].any(axis=1).sum()
print(f"\n  TOTAL rows with at least one flag: {total_flagged} ({total_flagged/len(df)*100:.1f}%)")

⚠️  Underage carers found (age < 18): 8
⚠️  years_caring capped for 248 rows
⚠️  Age-relationship mismatches: 50
⚠️  Implausible carer ages: 2
⚠️  Implausible PCF ages: 12
⚠️  Hours/week > 168: 26
   Hours/week distribution:
count    11697.000000
mean        63.345901
std        232.999547
min          0.000000
25%         49.000000
50%         50.000000
75%         70.000000
max      25000.000000
Name: hours_per_week, dtype: float64

📅 Assessment date range: 2018-03-26 00:00:00 to 2022-02-28 00:00:00
⚠️  Date outliers: 0
⚠️  Identical carer & PCF disabilities: 340
⚠️  Exact duplicate rows: 0
⚠️  Duplicate IDs: 0

Carer gender distribution:
carer_gender
Female               8617
Male                 3113
NaN                    30
Prefer Not To Say       4
Other                   2
Undisclosed             2
Non-Binary              1
Name: count, dtype: int64

PCF gender distribution:
pcf_gender
Female               5327
Male                 4913
NaN                  1524
Transgender Ftm

In [16]:
df.to_csv("temp.csv")

In [19]:
# ============================================================
# ACTUAL FIXES FOR FLAGGED ISSUES
# ============================================================

# ----------------------------------------------------------
# 14a. Age-Relationship mismatches → fix by swapping or nullifying
# ----------------------------------------------------------
# Where carer cares for "Parent" but pcf_age < carer_age,
# the ages were likely entered in the wrong fields (swapped)

def fix_age_relationship(row):
    """Attempt to fix swapped ages based on relationship logic."""
    rel = row.get('pcf_relationship')
    carer_age = row.get('carer_age')
    pcf_age = row.get('pcf_age')
    
    if pd.isna(rel) or pd.isna(carer_age) or pd.isna(pcf_age):
        return row
    
    rel_lower = str(rel).lower().strip()
    
    # Caring for parent but parent is younger → likely swapped
    if rel_lower in ['parent', 'mother', 'father']:
        if pcf_age < carer_age:
            row['carer_age'], row['pcf_age'] = row['pcf_age'], row['carer_age']
            row['flag_ages_swapped'] = True
    
    # Caring for child but child is older → likely swapped
    if rel_lower in ['son', 'daughter', 'child', 'son/daughter']:
        if pcf_age > carer_age:
            row['carer_age'], row['pcf_age'] = row['pcf_age'], row['carer_age']
            row['flag_ages_swapped'] = True
    
    return row

df['flag_ages_swapped'] = False
df = df.apply(fix_age_relationship, axis=1)
swapped_count = df['flag_ages_swapped'].sum()
print(f"🔧 Ages swapped to match relationship: {swapped_count}")

# For spouse/partner with extreme gap (>40 yrs) — can't determine 
# which is wrong, so nullify both
extreme_spouse_mask = (
    df['pcf_relationship'].str.lower().isin(
        ['spouse', 'partner', 'husband', 'wife']
    ) &
    (abs(df['carer_age'] - df['pcf_age']) > 40)
)
df.loc[extreme_spouse_mask, ['carer_age', 'pcf_age']] = np.nan
print(f"🔧 Extreme spouse age gaps nullified: {extreme_spouse_mask.sum()}")


# ----------------------------------------------------------
# 14b. Implausible ages → nullify
# ----------------------------------------------------------
implausible_carer = (df['carer_age'] < 18) | (df['carer_age'] > 105)
implausible_pcf = (df['pcf_age'] < 0) | (df['pcf_age'] > 115)

df.loc[implausible_carer, 'carer_age'] = np.nan
df.loc[implausible_pcf, 'pcf_age'] = np.nan
print(f"🔧 Implausible carer ages nullified: {implausible_carer.sum()}")
print(f"🔧 Implausible PCF ages nullified: {implausible_pcf.sum()}")


# ----------------------------------------------------------
# 14c. Hours per week > 168 → cap at 168
# ----------------------------------------------------------
over_168 = df['hours_per_week'] > 168
df.loc[over_168, 'hours_per_week'] = 168
print(f"🔧 Hours/week capped at 168: {over_168.sum()}")


# ----------------------------------------------------------
# 14d. Assessment date outliers → nullify
# ----------------------------------------------------------
reasonable_start = pd.Timestamp('2015-01-01')
today = pd.Timestamp('today')

date_outlier_mask = (
    (df['assessment_date'] < reasonable_start) | 
    (df['assessment_date'] > today)
)
df.loc[date_outlier_mask, 'assessment_date'] = pd.NaT
# Also re-derive the date parts
df['assessment_year'] = df['assessment_date'].dt.year
df['assessment_month'] = df['assessment_date'].dt.month
df['assessment_quarter'] = df['assessment_date'].dt.quarter
print(f"🔧 Date outliers nullified: {date_outlier_mask.sum()}")


# ----------------------------------------------------------
# 14e. Scores outside 1-5 → nullify
# ----------------------------------------------------------
score_num_cols = [c for c in df.columns if c.endswith('_score_num')]
total_score_fixes = 0
for col in score_num_cols:
    bad_mask = (df[col] < 1) | (df[col] > 5)
    count = bad_mask.sum()
    if count > 0:
        df.loc[bad_mask, col] = np.nan
        # Also nullify the corresponding label
        label_col = col.replace('_num', '_label')
        if label_col in df.columns:
            df.loc[bad_mask, label_col] = np.nan
        total_score_fixes += count
print(f"🔧 Out-of-range scores nullified: {total_score_fixes} (across {len(score_num_cols)} columns)")


# ----------------------------------------------------------
# 14f. Identical carer & PCF disabilities → flag only (keep data)
# ----------------------------------------------------------
# This is suspicious but we can't determine which is wrong,
# so we flag it for manual review rather than deleting
df['flag_identical_disabilities'] = (
    df['carer_disability'].notna() & 
    df['pcf_disability'].notna() & 
    (df['carer_disability'] == df['pcf_disability'])
)
print(f"🚩 Identical disabilities flagged (not altered): {df['flag_identical_disabilities'].sum()}")


# ----------------------------------------------------------
# 14g. Duplicate records → remove
# ----------------------------------------------------------
# Exact duplicates: drop
exact_dupes_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"🔧 Exact duplicate rows removed: {exact_dupes_before}")

# Same ID + same date: keep the LAST entry (likely the most recent/corrected)
id_date_dupes_before = df.duplicated(subset=['ID', 'assessment_date'], keep='last').sum()
df = df.drop_duplicates(
    subset=['ID', 'assessment_date'], keep='last'
).reset_index(drop=True)
print(f"🔧 Same ID + same date duplicates removed (kept last): {id_date_dupes_before}")


# ----------------------------------------------------------
# 14h. Clean boilerplate text from reason columns
# ----------------------------------------------------------
reason_cols = [c for c in df.columns if c.endswith('_reason')]

boilerplate_phrases = [
    r'Have you thought of any changes you could make\??',
    r'If you are receiving any welfare benefits, please provide details',
    r'No further actions could be considered\.',
]

for col in reason_cols:
    if col in df.columns:
        for phrase in boilerplate_phrases:
            df[col] = df[col].astype(str).str.replace(
                phrase, '', regex=True
            )
        # Clean up leftover whitespace and colons
        df[col] = (df[col].str.replace(r'^[\s:]+', '', regex=True)   # leading colons/spaces.str.replace(r'[\s:]+$', '', regex=True)    # trailing colons/spaces.str.replace(r'\s{2,}', ' ', regex=True)    # multiple spaces.str.strip()
        )
        # Re-nullify empty strings
        df[col] = df[col].replace(['', 'nan', 'None'], np.nan)

print(f"🔧 Boilerplate stripped from {len(reason_cols)} reason columns")


# ----------------------------------------------------------
# 14i. Gender cleanup
# ----------------------------------------------------------
# Standardise values
gender_map = {
    'Male': 'Male',
    'Female': 'Female',
    'M': 'Male',
    'F': 'Female',
    'Man': 'Male',
    'Woman': 'Female',
    'Non-Binary': 'Non-Binary',
    'Prefer Not To Say': np.nan,
    'Not Provided': np.nan,
    'Not Known': np.nan,
}

for col in ['carer_gender', 'pcf_gender']:
    if col in df.columns:
        df[col] = df[col].str.strip().str.title().map(gender_map)

print(f"🔧 Gender values standardised")
print(f"   Carer gender: {df['carer_gender'].value_counts(dropna=False).to_dict()}")


# ----------------------------------------------------------
# 14j. Working status cleanup
# ----------------------------------------------------------
working_map = {
    'Full Time': 'Full Time',
    'Full-Time': 'Full Time',
    'Part Time': 'Part Time',
    'Part-Time': 'Part Time',
    'Retired': 'Retired',
    'Not Working': 'Not Working',
    'Unemployed': 'Not Working',
    'Self Employed': 'Self Employed',
    'Self-Employed': 'Self Employed',
    'Voluntary Work': 'Voluntary Work',
    'Student': 'Student/Training',
    'Training': 'Student/Training',
    'In Education': 'Student/Training',
}

df['working_status'] = (df['working_status'].str.strip().str.title().map(working_map).fillna(df['working_status'])  # keep original if not in map
)
print(f"🔧 Working status standardised")
print(f"   Values: {df['working_status'].value_counts(dropna=False).to_dict()}")


# ============================================================
# 15. RE-RUN years_caring CAP (after age fixes/swaps)
# ============================================================
# Since we may have swapped or nullified ages, re-apply the cap
df['years_caring'] = df.apply(cap_years_caring, axis=1)
print(f"🔧 years_caring re-capped after age corrections")


# ============================================================
# 16. FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("CLEANUP COMPLETE — FINAL SUMMARY")
print("=" * 60)
print(f"Final shape: {df.shape}")
print(f"\nRemaining flags for manual review:")
flag_cols = [c for c in df.columns if c.startswith('flag_')]
for col in flag_cols:
    count = df[col].sum()
    if count > 0:
        print(f"  🚩 {col}: {count} rows")

print(f"\nNull counts in key columns:")
key_cols = [
    'carer_age', 'pcf_age', 'years_caring', 'hours_per_week',
    'assessment_date', 'carer_disability', 'pcf_disability',
    'carer_gender', 'working_status'
]
for col in key_cols:
    if col in df.columns:
        nulls = df[col].isna().sum()
        pct = nulls / len(df) * 100
        print(f"  {col}: {nulls} nulls ({pct:.1f}%)")

🔧 Ages swapped to match relationship: 45
🔧 Extreme spouse age gaps nullified: 5
🔧 Implausible carer ages nullified: 2
🔧 Implausible PCF ages nullified: 10
🔧 Hours/week capped at 168: 26
🔧 Date outliers nullified: 0
🔧 Out-of-range scores nullified: 0 (across 9 columns)
🚩 Identical disabilities flagged (not altered): 340
🔧 Exact duplicate rows removed: 0
🔧 Same ID + same date duplicates removed (kept last): 0
🔧 Boilerplate stripped from 9 reason columns
🔧 Gender values standardised
   Carer gender: {'Female': 8617, 'Male': 3113, nan: 38, 'Non-Binary': 1}
🔧 Working status standardised
   Values: {'Not Working': 6451, 'Retired': 2371, 'Part Time': 1515, 'Full Time': 1116, 'Student/Training': 258, nan: 58}
🔧 years_caring re-capped after age corrections

CLEANUP COMPLETE — FINAL SUMMARY
Final shape: (11769, 123)

Remaining flags for manual review:
  🚩 flag_age_relationship_mismatch: 50 rows
  🚩 flag_carer_age_implausible: 2 rows
  🚩 flag_pcf_age_implausible: 12 rows
  🚩 flag_hours_implausibl

In [20]:
# ============================================================
# 17. RESOLVE REMAINING FLAGS
# ============================================================

# ----------------------------------------------------------
# 17a. Drop 2 rows with implausible carer age
# ----------------------------------------------------------
rows_before = len(df)
df = df[~df['flag_carer_age_implausible']].reset_index(drop=True)
print(f"🗑️  Dropped {rows_before - len(df)} rows with implausible carer age")


# ----------------------------------------------------------
# 17b. Hours implausible — verify the cap already applied
# ----------------------------------------------------------
# We capped at 168 earlier, but let's confirm and re-flag
still_over = (df['hours_per_week'] > 168).sum()
if still_over > 0:
    df.loc[df['hours_per_week'] > 168, 'hours_per_week'] = 168
    print(f"🔧 Capped {still_over} remaining hours > 168")
else:
    print(f"✅ Hours already capped — no remaining implausible values")

# Update the flag
df['flag_hours_implausible'] = False


# ----------------------------------------------------------
# 17c. Identical disabilities — no action, keep flag for reference
# ----------------------------------------------------------
print(f"✅ Identical disabilities: kept as-is ({df['flag_identical_disabilities'].sum()} rows flagged for reference)")


# ----------------------------------------------------------
# 17d. Age-relationship mismatch — handle the REMAINING mismatches
#      (after the 45 swaps already applied)
# ----------------------------------------------------------
# Recalculate the mismatch flag post-swap to find the unresolved ones
df['flag_age_relationship_mismatch'] = df.apply(
    flag_age_relationship_mismatch, axis=1
)
remaining_mismatches = df['flag_age_relationship_mismatch'].sum()
print(f"🔍 Age-relationship mismatches remaining after swaps: {remaining_mismatches}")

# Nullify both ages for these unresolvable cases
df.loc[
    df['flag_age_relationship_mismatch'], 
    ['carer_age', 'pcf_age']
] = np.nan
print(f"🔧 Nullified carer_age & pcf_age for {remaining_mismatches} unresolvable mismatches")


# ----------------------------------------------------------
# 17e. Ages swapped — already fixed, flag is just an audit trail
# ----------------------------------------------------------
print(f"✅ Swapped ages: no further action ({df['flag_ages_swapped'].sum()} rows were corrected earlier)")


# ----------------------------------------------------------
# 17f. PCF age implausible — nullify pcf_age only
# ----------------------------------------------------------
pcf_implausible_mask = df['flag_pcf_age_implausible']
df.loc[pcf_implausible_mask, 'pcf_age'] = np.nan
print(f"🔧 Nullified pcf_age for {pcf_implausible_mask.sum()} implausible values")


# ----------------------------------------------------------
# 17g. Re-cap years_caring one final time (ages changed again)
# ----------------------------------------------------------
df['years_caring'] = df.apply(cap_years_caring, axis=1)
print(f"🔧 years_caring re-capped after final age corrections")


# ----------------------------------------------------------
# 17h. Regenerate binned columns where source values changed
# ----------------------------------------------------------
# Since we swapped/nullified some ages, the existing age bins 
# may be stale. Regenerate them.

def bin_carer_age(age):
    if pd.isna(age):
        return np.nan
    if age < 25:    return '18-24'
    if age < 35:    return '25-34'
    if age < 45:    return '35-44'
    if age < 55:    return '45-54'
    if age < 65:    return '55-64'
    if age < 75:    return '65-74'
    if age < 85:    return '75-84'
    if age < 95:    return '85-94'
    return '95+'

def bin_pcf_age(age):
    if pd.isna(age):
        return np.nan
    if age < 18:    return '0-17'
    if age < 25:    return '18-24'
    if age < 35:    return '25-34'
    if age < 45:    return '35-44'
    if age < 55:    return '45-54'
    if age < 65:    return '55-64'
    if age < 75:    return '65-74'
    if age < 85:    return '75-84'
    if age < 95:    return '85-94'
    return '95+'

df['carer_age_bin'] = df['carer_age'].apply(bin_carer_age)
df['pcf_age_bin'] = df['pcf_age'].apply(bin_pcf_age)
print(f"🔧 Age bins regenerated from corrected ages")


# ============================================================
# 18. CLEAN UP FLAG COLUMNS
# ============================================================
# Keep useful audit flags, drop resolved ones

flags_to_keep = [
    'flag_identical_disabilities',  # informational — for reference
    'flag_ages_swapped',            # audit trail — which rows were corrected
]

flags_to_drop = [
    'flag_age_relationship_mismatch',  # resolved (nullified)
    'flag_carer_age_implausible',      # resolved (rows dropped)
    'flag_pcf_age_implausible',        # resolved (nullified)
    'flag_hours_implausible',          # resolved (capped)
    'flag_date_outlier',               # resolved (nullified)
]

df.drop(columns=[c for c in flags_to_drop if c in df.columns], inplace=True)
print(f"\n🧹 Dropped resolved flag columns: {flags_to_drop}")
print(f"📌 Kept audit flag columns: {flags_to_keep}")


# ============================================================
# 19. FINAL VERIFICATION
# ============================================================
print("\n" + "=" * 60)
print("FINAL CLEAN DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df.shape}")

print(f"\nAge ranges (after all corrections):")
print(f"  Carer age:  {df['carer_age'].min():.0f} – {df['carer_age'].max():.0f}  "
      f"(nulls: {df['carer_age'].isna().sum()})")
print(f"  PCF age:    {df['pcf_age'].min():.0f} – {df['pcf_age'].max():.0f}  "
      f"(nulls: {df['pcf_age'].isna().sum()})")
print(f"  Years caring: {df['years_caring'].min():.0f} – {df['years_caring'].max():.0f}  "
      f"(nulls: {df['years_caring'].isna().sum()})")
print(f"  Hours/week: {df['hours_per_week'].min():.0f} – {df['hours_per_week'].max():.0f}  "
      f"(nulls: {df['hours_per_week'].isna().sum()})")

print(f"\nAssessment date range: {df['assessment_date'].min()} to {df['assessment_date'].max()}")

print(f"\nRemaining audit flags:")
for col in flags_to_keep:
    if col in df.columns:
        print(f"  📌 {col}: {df[col].sum()} rows")

🗑️  Dropped 2 rows with implausible carer age
✅ Hours already capped — no remaining implausible values
✅ Identical disabilities: kept as-is (340 rows flagged for reference)
🔍 Age-relationship mismatches remaining after swaps: 0
🔧 Nullified carer_age & pcf_age for 0 unresolvable mismatches
✅ Swapped ages: no further action (45 rows were corrected earlier)
🔧 Nullified pcf_age for 10 implausible values
🔧 years_caring re-capped after final age corrections
🔧 Age bins regenerated from corrected ages

🧹 Dropped resolved flag columns: ['flag_age_relationship_mismatch', 'flag_carer_age_implausible', 'flag_pcf_age_implausible', 'flag_hours_implausible', 'flag_date_outlier']
📌 Kept audit flag columns: ['flag_identical_disabilities', 'flag_ages_swapped']

FINAL CLEAN DATASET SUMMARY
Shape: (11767, 118)

Age ranges (after all corrections):
  Carer age:  18 – 104  (nulls: 14)
  PCF age:    18 – 110  (nulls: 683)
  Years caring: 0 – 60  (nulls: 902)
  Hours/week: 0 – 168  (nulls: 72)

Assessment date

In [22]:
df.to_csv("processed_data.csv")